# NumCompute-Stream: Streaming ML Demo

This notebook demonstrates the full streaming machine learning pipeline built in Assignment 2.2.

**What this demo covers:**
1. Load a dataset from CSV using our custom `io.py`
2. Split into chunks to simulate a streaming data setting
3. Train incrementally using `.partial_fit()` on each chunk
4. Compare a single Decision Tree vs Random Forest ensemble
5. Log and visualise key metrics over time using `visualise.py`

## 1. Imports and Setup

In [15]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from numcompute.io import load_csv_chunked, load_csv

from numcompute_stream.preprocessing import StandardScaler, SimpleImputer
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.ensemble import EnsembleClassifier
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.stream import StreamTrainer
from numcompute_stream.metrics import StreamingMetrics
from numcompute_stream.stats import StreamingStats
from numcompute_stream import visualise

print('All imports successful!')
print(f'NumPy version: {np.__version__}')

All imports successful!
NumPy version: 2.2.6


## 2. Load Dataset via Custom I/O Pipeline

In [16]:
DATA_PATH = 'iris_stream.csv'
CHUNK_SIZE = 30

data = load_csv(DATA_PATH)

print(f'Dataset shape : {data.shape}')
print(f'Features      : sepal_length, sepal_width, petal_length, petal_width')
print(f'Target        : species (0, 1, 2)')
print(f'Class counts  : {dict(zip(*np.unique(data[:, -1].astype(int), return_counts=True)))}')
print(f'\nFirst 5 rows:')
print(data[:5])

Dataset shape : (300, 5)
Features      : sepal_length, sepal_width, petal_length, petal_width
Target        : species (0, 1, 2)
Class counts  : {np.int64(0): np.int64(100), np.int64(1): np.int64(100), np.int64(2): np.int64(100)}

First 5 rows:
[[4.42 2.53 5.61 1.62 1.  ]
 [6.49 2.2  5.14 2.11 2.  ]
 [5.76 3.03 4.25 1.87 1.  ]
 [5.47 2.89 5.01 1.24 1.  ]
 [6.31 3.11 5.03 1.14 1.  ]]


## 3. Simulate Streaming: Split Into Chunks

In [17]:
chunks = []
for chunk in load_csv_chunked(DATA_PATH, chunksize=CHUNK_SIZE):
    X_chunk = chunk[:, :-1]
    y_chunk = chunk[:, -1].astype(int)
    chunks.append((X_chunk, y_chunk))

print(f'Total chunks  : {len(chunks)}')
print(f'Chunk size    : {CHUNK_SIZE} rows each')
print(f'Total samples : {sum(len(c[1]) for c in chunks)}')
print()
for i, (X_c, y_c) in enumerate(chunks):
    print(f'  Chunk {i+1:>2}: shape={X_c.shape}  classes={np.unique(y_c).tolist()}')

Total chunks  : 10
Chunk size    : 30 rows each
Total samples : 300

  Chunk  1: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  2: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  3: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  4: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  5: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  6: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  7: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  8: shape=(30, 4)  classes=[0, 1, 2]
  Chunk  9: shape=(30, 4)  classes=[0, 1, 2]
  Chunk 10: shape=(30, 4)  classes=[0, 1, 2]


## 4. Inspect Streaming Statistics

In [18]:
ss = StreamingStats(window_size=60)

for X_chunk, _ in chunks:
    ss.update(X_chunk)

stats = ss.summary_dict()
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

print(f'Samples seen: {stats["n_samples_seen"]}')
print()
print(f'{"Feature":<15} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 50)
for i, name in enumerate(feature_names):
    print(f'{name:<15} {stats["mean"][i]:>8.3f} {stats["std"][i]:>8.3f} '
          f'{stats["min"][i]:>8.3f} {stats["max"][i]:>8.3f}')

Samples seen: 300

Feature             Mean      Std      Min      Max
--------------------------------------------------
sepal_length       4.812    2.079    0.850    7.550
sepal_width        3.054    0.561    1.180    4.450
petal_length       3.795    1.804    0.240    7.190
petal_width        1.258    0.797    0.100    2.980


## 5. Build Pipelines

In [19]:
pipe_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('model',   DecisionTreeClassifier(max_depth=4, criterion='gini')),
])

pipe_rf = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('model',   EnsembleClassifier(n_estimators=15, method='random_forest', max_depth=4)),
])

print('Pipeline 1 — Single Decision Tree:')
print(pipe_tree)
print()
print('Pipeline 2 — Random Forest (15 trees):')
print(pipe_rf)

Pipeline 1 — Single Decision Tree:
Pipeline([imputer(SimpleImputer) -> scaler(StandardScaler) -> model(DecisionTreeClassifier)])

Pipeline 2 — Random Forest (15 trees):
Pipeline([imputer(SimpleImputer) -> scaler(StandardScaler) -> model(EnsembleClassifier)])


## 6. Incremental Training — Chunk by Chunk

In [20]:
trainer_tree = StreamTrainer(pipeline=pipe_tree, verbose=True)
trainer_rf   = StreamTrainer(pipeline=pipe_rf,   verbose=True)

all_classes = np.array([0, 1, 2])

print('=== Decision Tree ===' )
for X_chunk, y_chunk in chunks:
    trainer_tree.fit_chunk(X_chunk, y_chunk, classes=all_classes)
    trainer_tree.score_chunk(X_chunk, y_chunk)

print()
print('=== Random Forest ===')
for X_chunk, y_chunk in chunks:
    trainer_rf.fit_chunk(X_chunk, y_chunk, classes=all_classes)
    trainer_rf.score_chunk(X_chunk, y_chunk)

=== Decision Tree ===
[Chunk   1] n=30    | fit=135.5ms | mem=12.9KB | total_seen=30
           chunk_acc=0.9667 | cumul_acc=0.9667
[Chunk   2] n=30    | fit=234.9ms | mem=19.5KB | total_seen=60
           chunk_acc=1.0000 | cumul_acc=0.9833
[Chunk   3] n=30    | fit=446.6ms | mem=23.7KB | total_seen=90
           chunk_acc=0.9333 | cumul_acc=0.9667
[Chunk   4] n=30    | fit=612.8ms | mem=25.1KB | total_seen=120
           chunk_acc=1.0000 | cumul_acc=0.9750
[Chunk   5] n=30    | fit=876.0ms | mem=26.0KB | total_seen=150
           chunk_acc=1.0000 | cumul_acc=0.9800
[Chunk   6] n=30    | fit=907.1ms | mem=27.9KB | total_seen=180
           chunk_acc=0.9667 | cumul_acc=0.9778
[Chunk   7] n=30    | fit=1095.6ms | mem=31.3KB | total_seen=210
           chunk_acc=1.0000 | cumul_acc=0.9810
[Chunk   8] n=30    | fit=1397.0ms | mem=34.9KB | total_seen=240
           chunk_acc=1.0000 | cumul_acc=0.9833
[Chunk   9] n=30    | fit=1554.4ms | mem=38.4KB | total_seen=270
           chunk_acc=0.966

## 7. Training Summary

In [21]:
print('Decision Tree Summary:')
for k, v in trainer_tree.summary().items():
    print(f'  {k:<35} {v}')

print()
print('Random Forest Summary:')
for k, v in trainer_rf.summary().items():
    print(f'  {k:<35} {v}')

print()
print('Decision Tree Full Metrics:')
result = trainer_tree.metrics.result()
for k, v in result.items():
    if k != 'confusion_matrix':
        print(f'  {k:<20} {v}')

print()
print('Random Forest Full Metrics:')
result_rf = trainer_rf.metrics.result()
for k, v in result_rf.items():
    if k != 'confusion_matrix':
        print(f'  {k:<20} {v}')

Decision Tree Summary:
  n_chunks                            10
  n_samples                           300
  total_fit_time_ms                   9264.98
  mean_chunk_accuracy                 0.9767
  final_cumulative_accuracy           0.9767

Random Forest Summary:
  n_chunks                            10
  n_samples                           300
  total_fit_time_ms                   63679.11
  mean_chunk_accuracy                 0.9733
  final_cumulative_accuracy           0.9733

Decision Tree Full Metrics:
  accuracy             0.9766666666666667
  precision            0.9766976697669767
  recall               0.9766666666666666
  f1                   0.9766821679707866
  n_samples_seen       300
  n_chunks_seen        10

Random Forest Full Metrics:
  accuracy             0.9733333333333334
  precision            0.9733333333333333
  recall               0.9733333333333333
  f1                   0.9733333333333333
  n_samples_seen       300
  n_chunks_seen        10


## 8. Visualisation — Accuracy Over Time

In [22]:
tree_acc_history = trainer_tree.get_accuracy_history()
rf_acc_history   = trainer_rf.get_accuracy_history()

visualise.plot_metric_over_time(
    tree_acc_history,
    title='Decision Tree — Accuracy over Streaming Chunks',
    ylabel='Accuracy',
    color='#4C72B0',
    rolling_window=3,
    save_path='tree_accuracy.png',
)

print('Saved: tree_accuracy.png')

Saved: tree_accuracy.png


## 9. Visualisation — Model Comparison

In [23]:
visualise.compare_models(
    tree_acc_history,
    rf_acc_history,
    labels=('Decision Tree', 'Random Forest'),
    title='Decision Tree vs Random Forest — Streaming Accuracy',
    ylabel='Accuracy',
    save_path='model_comparison.png',
)

print('Saved: model_comparison.png')

Saved: model_comparison.png


## 10. Visualisation — Predictions vs Ground Truth (Latest Chunk)

In [24]:
X_last, y_last = chunks[-1]
y_pred_tree = pipe_tree.predict(X_last)
y_pred_rf   = pipe_rf.predict(X_last)

print(f'Latest chunk — {len(y_last)} samples')
print(f'Decision Tree accuracy : {np.mean(y_pred_tree == y_last):.4f}')
print(f'Random Forest accuracy : {np.mean(y_pred_rf == y_last):.4f}')
print()

visualise.plot_predictions_vs_ground_truth(
    y_last,
    y_pred_rf,
    title='Random Forest — Predictions vs Ground Truth (Latest Chunk)',
    save_path='predictions_vs_truth.png',
)

print('Saved: predictions_vs_truth.png')

Latest chunk — 30 samples
Decision Tree accuracy : 0.9333
Random Forest accuracy : 0.9667

Saved: predictions_vs_truth.png


## 11. Visualisation — Confusion Matrix

In [25]:
cm = trainer_rf.metrics.get_confusion_matrix()

visualise.plot_confusion_matrix(
    cm,
    title='Random Forest — Cumulative Confusion Matrix',
    save_path='confusion_matrix.png',
)

print('Saved: confusion_matrix.png')

Saved: confusion_matrix.png


## 12. Visualisation — Training Time per Chunk

In [26]:
visualise.plot_fit_times(
    trainer_rf.get_fit_times(),
    title='Random Forest — Fit Time per Chunk (ms)',
    save_path='fit_times.png',
)

print('Saved: fit_times.png')

Saved: fit_times.png


## 13. Cumulative Accuracy Over Time

In [27]:
visualise.compare_models(
    trainer_tree.get_cumulative_accuracy_history(),
    trainer_rf.get_cumulative_accuracy_history(),
    labels=('Decision Tree (cumulative)', 'Random Forest (cumulative)'),
    title='Cumulative Accuracy over Streaming Chunks',
    ylabel='Cumulative Accuracy',
    save_path='cumulative_accuracy.png',
)

print('Saved: cumulative_accuracy.png')

Saved: cumulative_accuracy.png


## 14. Final Results

In [28]:
print('=' * 50)
print('FINAL RESULTS')
print('=' * 50)

tree_final = trainer_tree.metrics.get_accuracy()
rf_final   = trainer_rf.metrics.get_accuracy()

print(f'Decision Tree final accuracy  : {tree_final:.4f} ({tree_final*100:.1f}%)')
print(f'Random Forest final accuracy  : {rf_final:.4f} ({rf_final*100:.1f}%)')
print(f'Improvement from ensemble     : {(rf_final - tree_final)*100:+.1f}%')
print()
print(f'Total chunks processed        : {len(chunks)}')
print(f'Total samples seen            : {trainer_rf.n_samples_trained_}')
print(f'Total RF fit time             : {trainer_rf.summary()["total_fit_time_ms"]:.1f}ms')
print()
print('Plots saved to demo/ folder:')
for fname in ['tree_accuracy.png', 'model_comparison.png', 'predictions_vs_truth.png',
              'confusion_matrix.png', 'fit_times.png', 'cumulative_accuracy.png']:
    print(f'  {fname}')

FINAL RESULTS
Decision Tree final accuracy  : 0.9767 (97.7%)
Random Forest final accuracy  : 0.9733 (97.3%)
Improvement from ensemble     : -0.3%

Total chunks processed        : 10
Total samples seen            : 300
Total RF fit time             : 63679.1ms

Plots saved to demo/ folder:
  tree_accuracy.png
  model_comparison.png
  predictions_vs_truth.png
  confusion_matrix.png
  fit_times.png
  cumulative_accuracy.png
